# V15.5 continuation 01: offline publication checks

## tl;dr

Share with caveats: the continuation halted after 155 valid diagnoses and four
ambiguous timeouts; 161 requests were never sent. Combined main coverage is
343/512, not a complete study. OpenAI remains blocked and the old gate remains
FAIL. This notebook verifies accounting, provenance, literal outputs and costs;
it does not certify model diagnoses as human gold or authorize a learner run.

## Context & Methods

As of 2026-10-10, Asia/Tokyo. The 512 original outputs cover 16 correlated world
clusters; 32 repeats are measurements of fixed outputs, not extra independent
tasks. Original paid source: `59ac10623c0f088d61481e1cfbb9898c206002b5`;
continuation source: `92bd66a0024f8c4b4442d1addfa4105f60287b96`.

### Key Assumptions

The original and continuation bundles remain accessible in this repository.
Remote weight revision and actual invoice are unknown. Missing observations
are not assumed random. Frozen buckets are hierarchical, not independent
dimensions. No new SQL source, credential, network or inference is needed.

## Data

Sources: this bundle's `analysis/{SUMMARY,LEDGER,ARTIFACT_INDEX}.json`, its raw
call receipts, and the immutable sibling `v15_5_diagnostic_judge_20261010`.
Run from anywhere inside the checkout. All cells are read-only. The repository
`.venv` is unchanged; notebook tooling uses an isolated `uv` environment.

### 1. Load and replay the sealed artifacts


In [1]:
import json
import sys
from collections import Counter
from decimal import Decimal
from pathlib import Path

repo = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'scripts/continue_v15_5_diagnostic_judge.py').is_file())
sys.path.insert(0, str(repo / 'scripts'))
import continue_v15_5_diagnostic_judge as continuation
import run_v15_5_diagnostic_judge as original_runner
import v15_5_diagnostics as diagnostics

root = repo / continuation.ROOT
original_root = repo / continuation.ORIGINAL
receipt = continuation.analysis_artifacts(verify=True)
summary = json.loads((root / 'analysis/SUMMARY.json').read_text())
ledger = json.loads((root / 'analysis/LEDGER.json').read_text())
old_summary = json.loads((original_root / 'analysis/SUMMARY.json').read_text())
plan, data, _seal = original_runner.load_frozen()
assert len(ledger) == len({r['item_id'] for r in ledger}) == 512
assert len({r['family_id'] for r in ledger}) == 16
assert summary['status'] == 'INCOMPLETE_OR_CALIBRATION_BLOCKED'
assert summary['old_expression_gate'] == 'FAIL'
assert summary['providers']['openai'] == old_summary['providers']['openai']
print(json.dumps({'source_files': summary['source_files_verified'],
    'original_bundle_files': summary['continuation']['original_bundle_files_verified'],
    'ledger_rows': len(ledger), 'world_clusters': 16, 'old_gate': 'FAIL'}, sort_keys=True))


{"ledger_rows": 512, "old_gate": "FAIL", "original_bundle_files": 1042, "source_files": 79, "world_clusters": 16}


## Results

### 2. Independently account for every planned request

Read raw receipts from both bundles, reject duplicate request IDs, and compare
their union with the original 1,120-coordinate plan. Unknown execution stays
separate from never dispatched; calibration is not counted twice.


In [2]:
planned = {r['request_id']: r
    for provider in original_runner.PROVIDERS
    for phase in ('calibration', 'study')
    for r in diagnostics.requests_for(data, plan, provider, phase)}
seen = {}
for source in (original_root, root):
    for request_path in sorted(source.glob('*/study/calls/*/REQUEST.json')) + sorted(
            source.glob('*/calibration/calls/*/REQUEST.json')):
        request = json.loads(request_path.read_text())
        request_id = request['request_id']
        assert request_id in planned and request == planned[request_id]
        assert request_id not in seen, 'Duplicate paid coordinate across bundles'
        outcome = json.loads((request_path.parent / 'OUTCOME.json').read_text())
        status = (outcome['observation']['status'] if outcome['status'] == 'response_recorded'
                  else outcome['status'])
        seen[request_id] = (request, outcome, status, source.name)
counts = Counter(value[2] for value in seen.values())
counts['not_dispatched'] = len(planned) - len(seen)
assert len(planned) == 1120 and len(seen) == 415
assert dict(counts) == {'valid_diagnosis': 406, 'invalid_judgment': 1,
    'transport_or_recording_ambiguous_no_retry': 8, 'not_dispatched': 705}
assert sum(counts.values()) == 1120
assert all(not key.startswith('openai-study-') for key in seen)
new = [value for value in seen.values() if value[3] == root.name]
assert len(new) == 159
assert Counter(value[2] for value in new) == {'valid_diagnosis': 155,
    'transport_or_recording_ambiguous_no_retry': 4}
assert all((value[0]['provider'], value[0]['phase'], value[0]['replicate'])
           == ('mistral', 'study', 0) for value in new)
print(json.dumps({'all_phases': dict(counts), 'reserved': len(seen),
    'new_reserved': len(new), 'new_never_dispatched': 320-len(new)}, sort_keys=True))


{"all_phases": {"invalid_judgment": 1, "not_dispatched": 705, "transport_or_recording_ambiguous_no_retry": 8, "valid_diagnosis": 406}, "new_never_dispatched": 161, "new_reserved": 159, "reserved": 415}


### 3. Reconcile denominators and renderer-specific failure categories

These are descriptive model diagnoses, not corrected scores. Separate raw from
native-chat formatting before attributing failure to termination. Preserve all
512 machine outputs, including those lacking a valid model diagnosis.


In [3]:
profile = {}
for information in ('inline', 'query_only'):
    rows = [r for r in ledger if r['coordinate']['information'] == information]
    assert len(rows) == 256
    status = Counter(r['diagnoses']['mistral']['status'] for r in rows)
    buckets = Counter(r['diagnoses']['mistral']['bucket'] for r in rows)
    section = summary['primary_inline' if information == 'inline' else 'secondary_query_only']
    assert dict(status) == section['judges']['mistral']['statuses']
    assert dict(buckets) == section['judges']['mistral']['buckets']
    assert sum(buckets.values()) == len(rows)
    assert sum(r['mechanical']['strict_correct'] for r in rows) == section['strict_correct']
    profile[information] = {'planned': len(rows), 'statuses': dict(status),
        'strict_all': section['strict_correct'], 'buckets': dict(buckets)}
native = [r for r in ledger if r['coordinate']['information'] == 'inline'
          and r['coordinate']['renderer'] == 'native_chat']
valid_native = [r for r in native if r['diagnoses']['mistral']['status'] == 'valid_diagnosis']
native_buckets = Counter(r['diagnoses']['mistral']['bucket'] for r in valid_native)
assert len(native) == 128 and len(valid_native) == 87
assert dict(native_buckets) == {'strict_success': 58, 'internal_contradiction': 13,
    'unambiguous_wrong_commitment': 13, 'format_only_candidate_not_rescued': 2,
    'explanation_or_grounding_failure': 1}
assert sum(not r['mechanical']['strict_correct'] for r in valid_native) == 29
print(json.dumps({'information': profile, 'native_inline_valid': len(valid_native),
    'native_inline_valid_buckets': dict(native_buckets)}, sort_keys=True))


{"information": {"inline": {"buckets": {"explanation_or_grounding_failure": 1, "format_only_candidate_not_rescued": 4, "incomplete_observed_text": 82, "internal_contradiction": 13, "strict_success": 58, "unambiguous_wrong_commitment": 13, "unavailable_judgment": 85}, "planned": 256, "statuses": {"not_dispatched": 81, "transport_or_recording_ambiguous_no_retry": 4, "valid_diagnosis": 171}, "strict_all": 84}, "query_only": {"buckets": {"abstention": 11, "explanation_or_grounding_failure": 11, "incomplete_observed_text": 109, "internal_contradiction": 31, "strict_success": 1, "unambiguous_wrong_commitment": 9, "unavailable_judgment": 84}, "planned": 256, "statuses": {"not_dispatched": 80, "transport_or_recording_ambiguous_no_retry": 4, "valid_diagnosis": 172}, "strict_all": 1}}, "native_inline_valid": 87, "native_inline_valid_buckets": {"explanation_or_grounding_failure": 1, "format_only_candidate_not_rescued": 2, "internal_contradiction": 13, "strict_success": 58, "unambiguous_wrong_comm

### 4. Recompute usage from receipts and preserve repeat disagreement

Costs use frozen undiscounted plan rates, not a current price claim or invoice.
The eight timed-out calls have unknown remote execution and billing.


In [4]:
usage = {}
for provider in ('mistral', 'openai'):
    values = [value for value in seen.values() if value[0]['provider'] == provider]
    known = [value[1]['usage_receipt'] for value in values
             if value[1].get('usage_receipt', {}).get('status') == 'REPORTED_BY_API']
    actual = {'reserved': len(values), 'reported': len(known),
        'input': sum(r['input_tokens'] for r in known),
        'output': sum(r['output_tokens'] for r in known),
        'cost': str(sum((Decimal(r['usage_based_undiscounted_cost_usd'])
                         for r in known), Decimal(0)))}
    reported = summary['providers'][provider]['usage']
    observed = tuple(actual[k] for k in ('reserved', 'reported', 'input', 'output', 'cost'))
    assert observed == (
        reported['reserved_calls'], reported['calls_with_reported_usage'],
        reported['reported_input_tokens'], reported['reported_output_tokens'],
        reported['reported_usage_undiscounted_cost_usd'])
    usage[provider] = actual
assert sum(v['reported'] for v in usage.values()) == 407
cost = sum(Decimal(v['cost']) for v in usage.values())
assert cost == Decimal('7.73465540')
repeat = summary['providers']['mistral']['repeat_stability']
assert repeat == old_summary['providers']['mistral']['repeat_stability']
assert (repeat['all_four_equal'], repeat['differs'], repeat['invalid_or_missing']) == (25, 6, 1)
print(json.dumps({'usage': usage, 'cost_total': str(cost), 'repeat': repeat}, sort_keys=True))


{"cost_total": "7.73465540", "repeat": {"all_four_equal": 25, "differs": 6, "dimensions": {"added_facts": {"differs": 5, "equal": 26}, "commitment": {"equal": 31}, "contradiction": {"differs": 1, "equal": 30}, "reasoning": {"differs": 3, "equal": 28}}, "invalid_or_missing": 1, "planned": 32}, "usage": {"mistral": {"cost": "7.58989540", "input": 426847, "output": 1676886, "reported": 391, "reserved": 399}, "openai": {"cost": "0.1447600", "input": 15034, "output": 7145, "reported": 16, "reserved": 16}}}


### 5. Verify publication examples, links and literal whitespace

The report's four quoted examples are explicitly selected illustrations, not a
representative sample. All code fences below are matched exactly to stored
learner text. Literal trailing spaces in the large generated panel are retained.


In [5]:
import re

report_path = root / 'README.md'
report_text = report_path.read_text()
example_ids = ('89f3d60346a5a601b10c3d20', '9db60525663ff106754b8281',
               '6cdf8806a56e76f4b6c15f97', '330a2cbcd94d10a9b4678dc7')
by_id = {r['item_id']: r for r in ledger}
fences = re.findall(r'```text\n(.*?)\n```', report_text, re.S)
assert len(fences) == len(example_ids) == 4
for item_id, literal in zip(example_ids, fences, strict=True):
    assert item_id in report_text and literal == by_id[item_id]['payload']['response_text']
paths = (report_path, root/'EXECUTION_HANDOFF.md', repo/'README.md', repo/'docs/v15/NEXT_STEPS.md')
links = [(p, link) for p in paths for link in re.findall(r'\]\(([^)]+)\)', p.read_text())
         if not re.match(r'[a-z]+:', link)]
targets = [(p.parent/link.split('#')[0]).resolve() for p, link in links]
# The notebook self-link is materialized immediately after this execution.
assert all(p.exists() or p == (root/'VALIDATION.ipynb').resolve() for p in targets)
panel_lines = (root/'analysis/DIAGNOSTIC_PANEL.md').read_text().splitlines()
whitespace = [i for i, line in enumerate(panel_lines, 1)
              if line.endswith((' ', '\t'))]
print(json.dumps({'literal_examples': len(fences), 'valid_local_links': len(links),
    'preserved_panel_trailing_whitespace_lines': whitespace}, sort_keys=True))


{"literal_examples": 4, "preserved_panel_trailing_whitespace_lines": [7024, 13765, 16279], "valid_local_links": 59}


## Takeaways

Mechanical replay and accounting pass, but publication is **share with caveats**:
343/512 main diagnoses are available, eight calls remain ambiguous, and 161
Mistral plus 544 OpenAI study coordinates are undispatched. The original invalid
repeat remains invalid. Fixed-repeat disagreement and nonrandom missingness
prevent gold-label and corpus-wide prevalence claims. No substantive quality,
latent-state mechanism, learner improvement or non-regression is established.

Of 87 diagnosed native-chat inline outputs, 29 fail strict scoring: 13 internal
contradictions, 13 unambiguous wrong commitments, two format-only candidates and
one explanation/grounding failure. A termination-only explanation is insufficient
for these observed failures. This motivates separate answer-content and stopping
checks, not immediate training or a repaired old FAIL.
